In [1]:
# This code demonstrates how the data flows through nodes and how messages with metadata are stored #and updated in the state We’ll build a two-node workflow:
#Node 1 – Summarizer: Takes user input → creates a summary → adds an annotated AIMessage.
#Node 2 – Validator: Checks if the summary length is acceptable → appends another annotated message.
#Finally, the workflow will return the full conversation state with all annotated messages.
# -----------------------------
# Imports
# -----------------------------
from typing import TypedDict, List
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
# -----------------------------
#  Step 1: Define the State
# -----------------------------
class SummarizationState(TypedDict):
    user_input: str
    summary: str
    messages: List  # This will hold annotated messages
# -----------------------------
# 🧩 Step 2: Define the Nodes
# -----------------------------
# Node 1: Summarizer
def summarizer_node(state: SummarizationState):
    text = state["user_input"]
    summary = text[:60] + "..."  # simple placeholder summary
    # Annotate the AI message
    ai_msg = AIMessage(
        content=f"Summary: {summary}",
        metadata={"node": "summarizer", "type": "summary"}
    )
    # Update the state
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["summary"] = summary
    state["messages"] = messages
    return state
# Node 2: Validator
def validator_node(state: SummarizationState):
    summary = state["summary"]
    valid = len(summary) < 80
    validation_result = "Valid summary" if valid else "Too long summary"
    ai_msg = AIMessage(
        content=validation_result,
        metadata={"node": "validator", "type": "validation"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# -----------------------------
#  Step 3: Create the Graph
# -----------------------------
graph = StateGraph(SummarizationState)
graph.add_node("summarizer", summarizer_node)
graph.add_node("validator", validator_node)
graph.add_edge(START, "summarizer")
graph.add_edge("summarizer", "validator")
graph.add_edge("validator", END)
# Compile the graph
app = graph.compile()
# -----------------------------
# Step 4: Run the Graph
# -----------------------------
input_text = "LangGraph simplifies building stateful AI agents using Python graphs."
initial_state = {
    "user_input": input_text,
    "summary": "",
    "messages": [HumanMessage(content=input_text)]
}
# Run the workflow
final_state = app.invoke(initial_state)
# -----------------------------
#  Step 5: Observe Results
# -----------------------------
print("=== Final State ===")
print(final_state)
print("\n=== Annotated Messages ===")
for msg in final_state["messages"]:
    meta = getattr(msg, "metadata", {})  # Use empty dict if not present
    print(f"{msg.type.upper()} -> {msg.content} | metadata={meta}")


=== Final State ===
{'user_input': 'LangGraph simplifies building stateful AI agents using Python graphs.', 'summary': 'LangGraph simplifies building stateful AI agents using Pytho...', 'messages': [HumanMessage(content='LangGraph simplifies building stateful AI agents using Python graphs.', additional_kwargs={}, response_metadata={}), AIMessage(content='Summary: LangGraph simplifies building stateful AI agents using Pytho...', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[], metadata={'node': 'summarizer', 'type': 'summary'}), AIMessage(content='Valid summary', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[], metadata={'node': 'validator', 'type': 'validation'})]}

=== Annotated Messages ===
HUMAN -> LangGraph simplifies building stateful AI agents using Python graphs. | metadata={}
AI -> Summary: LangGraph simplifies building stateful AI agents using Pytho... | metadata={'node': 'summarizer', 'type': 'summary'}
AI -> V

/home/labuser/.local/lib/python3.10/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
# -----------------------------
# Imports
# -----------------------------
from typing import TypedDict, List
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
import os
# -----------------------------
# Step 2: Initialize the AWS Bedrock Model
# -----------------------------
os.environ["AWS_ACCESS_KEY_ID"]=""
os.environ["AWS_SECRET_ACCESS_KEY"]= ""
# ---------------------------
# AWS Bedrock LLM Setup
# ---------------------------
llm = ChatBedrockConverse(model_id="amazon.nova-lite-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=100)
#llm = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=100)


In [6]:
# -----------------------------
# Step 1: Define the State
# -----------------------------
class AgentState(TypedDict):
    user_input: str
    summary: str
    messages: List  # Annotated message history
    decision: str   # "summarize", "validate", or "end"
# -----------------------------
# Step 2: Define Nodes
# -----------------------------
# Node 1 - LLM decides what to do next
def controller_node(state: AgentState):
    """Ask the LLM: what should be done next?"""
    user_input = state["user_input"]
    messages = state.get("messages", [])
    prompt = f"""
You are an AI workflow controller.
User said: {user_input}
Decide the next step in the workflow.
Options:
1. summarize - if user input is long text
2. validate - if there is already a summary
3. end - if summary is valid or no action needed
Answer with ONLY one word: summarize, validate, or end.
"""
    decision = llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    state["decision"] = decision
    return state
# Node 3 - Summarization
def summarizer_node(state: AgentState):
    text = state["user_input"]
    summary = llm.invoke([
        HumanMessage(content=f"Summarize this text briefly:\n\n{text}")
    ]).content.strip()
    state["summary"] = summary
    ai_msg = AIMessage(
        content=f"Summary: {summary}",
        metadata={"node": "summarizer"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# Node 4 - Validator
def validator_node(state: AgentState):
    summary = state["summary"]
    validation = llm.invoke([
        HumanMessage(content=f"Is the following summary under 80 words? Reply only with Yes or No.\n\n{summary}")
    ]).content.strip()
    ai_msg = AIMessage(
        content=f"Validation result: {validation}",
        metadata={"node": "validator"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# -----------------------------
# Step 5: Build the Graph
# -----------------------------
graph = StateGraph(AgentState)
graph.add_node("controller", controller_node)
graph.add_node("summarizer", summarizer_node)
graph.add_node("validator", validator_node)
# Start → Controller
graph.add_edge(START, "controller")
# Conditional transitions based on decision
graph.add_conditional_edges(
    "controller",
    lambda state: state["decision"],
    {
        "summarize": "summarizer",
        "validate": "validator",
        "end": END,
    }
)
# After summarization → validation
graph.add_edge("summarizer", "validator")
# After validation → End
graph.add_edge("validator", END)
# Compile
app = graph.compile()
# -----------------------------
# Step 7: Run the Agent
# -----------------------------
input_text = """
LangGraph is a Python framework that helps developers build complex,
stateful, multi-step AI agents using a directed graph of nodes.
"""
initial_state = {
    "user_input": input_text,
    "summary": "",
    "messages": [HumanMessage(content=input_text)],
    "decision": ""
}
# Run the dynamic graph
final_state = app.invoke(initial_state)
# -----------------------------
# Step 7: Print Results
# -----------------------------
print("=== Final State ===")
print(final_state)
print("\n=== Annotated Messages ===")
for msg in final_state["messages"]:
    meta = getattr(msg, "metadata", {})  # Use empty dict if not present
    print(f"{msg.type.upper()} -> {msg.content} | metadata={meta}")


=== Final State ===
{'user_input': '\nLangGraph is a Python framework that helps developers build complex,\nstateful, multi-step AI agents using a directed graph of nodes.\n', 'summary': 'LangGraph is a Python framework designed to assist developers in constructing intricate, state-aware, multi-step AI agents through a directed graph of interconnected nodes.', 'messages': [HumanMessage(content='\nLangGraph is a Python framework that helps developers build complex,\nstateful, multi-step AI agents using a directed graph of nodes.\n', additional_kwargs={}, response_metadata={}), AIMessage(content='Summary: LangGraph is a Python framework designed to assist developers in constructing intricate, state-aware, multi-step AI agents through a directed graph of interconnected nodes.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[], metadata={'node': 'summarizer'}), AIMessage(content='Validation result: No.', additional_kwargs={}, response_metadata={}, tool_calls